# KGGen ingest v2 — COSORA (provenance + chunk)

Notebook de **experimento** para regeneración completa del grafo (plan v2).

**Plan:** [`docs/plan_graphrag_v2.md`](../../docs/plan_graphrag_v2.md)

### Colab + Drive (convención del proyecto)

| Recurso | Colab |
|---------|-------|
| `.env` | `MyDrive/variablentorno/.env` |
| Actas | `MyDrive/RAG_UPC_Final_project/` |
| Chroma | `.../RAG_UPC_Final_project/chroma_db` |
| Grafos | `.../RAG_UPC_Final_project/graph/` |

Ejecutar con `drive.mount` (celda 0). Datos y artefactos en Drive, no en el runtime efímero de Colab.

| Flag | Default | Descripción |
|------|---------|-------------|
| `SCHEMA_VERSION` | `2` | JSON con provenance |
| `CHUNK_STRATEGY` | `recursive` | Alineado con Chroma prod |
| `KG_CLUSTER` | `True` | Entity resolution KGGen |
| `KG_CLUSTER_FALLBACK` | `True` | Si cluster falla → seguir sin cluster (no pierde el batch) |
| `FORCE_REGEN` | `True` | Regenerar aunque exista JSON |
| `REINDEX_CHROMA` | `False` | **Fase 1:** `True` → reindex completo |
| `RUN_GRAPH_EXTRACT` | `True` | **Fase 2+3:** KGGen por chunk |
| `RUN_ALL_BATCHES` | `True` | **Fase 3:** todos los batches |
| `SKIP_COMPLETED_BATCHES` | `True` | Reanudar: skip JSON existente si `FORCE_REGEN=False` |
| `BATCH_IDS` | `None` | Opcional: `[3, 4]` solo esos batches |
| `BATCH_ID` | `1` | Solo si `RUN_ALL_BATCHES=False` |

**Fases:** 0 setup → 1 utils → 2 extract → 3 inventario → 4 reindex Chroma → 5 validación → 6 schema → **7 KGGen por chunk** → 8 catalog v2

Ejecutar **4 + 7** en la misma sesión Colab (`REINDEX_CHROMA=True`, `RUN_GRAPH_EXTRACT=True`) antes de probar.


## 0. Setup

In [2]:
%pip install -q kg-gen python-dotenv python-docx pandas langchain-text-splitters chromadb sentence-transformers torch rank_bm25

import shutil
import subprocess
if shutil.which("antiword") is None:
    try:
        subprocess.run(["apt-get", "install", "-y", "-q", "antiword"], check=False)
    except Exception:
        print("⚠️  antiword no instalado — solo .docx")

import json
import os
import re
import sys
import unicodedata
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
RUNTIME = "colab" if IN_COLAB else "local"

# ─── Flags v2 (plan Fase 0+) ─────────────────────────────────────────────
SCHEMA_VERSION = 2
CHUNK_STRATEGY = "recursive"       # alineado con Chroma prod (cosora_actas_e5)
CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
COLLECTION_NAME = "cosora_actas_e5"

KG_SOURCE = "original"
BATCH_SIZE = 10
BATCH_ID = 6
RUN_ALL_BATCHES = False             # Fase 3: True → todos los batches
SKIP_COMPLETED_BATCHES = True      # resume: skip JSON existente si FORCE_REGEN=False
BATCH_IDS = None                   # ej. [3, 4] para reanudar batches concretos
FORCE_REGEN = False
REINDEX_CHROMA = False             # Fase 1: True para reindex completo
RUN_GRAPH_EXTRACT = True           # Fase 2+3: KGGen por chunk

KG_MODEL = "openai/gpt-4o-mini"
KG_CONTEXT = (
    "Actas de obra ferroviaria en España. Personas, empresas (UTE, DF, DEO), "
    "elementos constructivos, incidencias, fechas."
)
KG_CLUSTER = True
KG_CLUSTER_FALLBACK = True       # batch 2+ puede fallar cluster → continúa sin fusionar
KG_CHUNK_SIZE = 5000               # tamaño interno kg-gen si el chunk Chroma es largo

RAW_DOCS_PATH_OVERRIDE = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

from dotenv import load_dotenv
if RUNTIME == "colab":
    load_dotenv("/content/drive/MyDrive/variablentorno/.env")
elif Path(".env").exists():
    load_dotenv(".env")
elif Path("../../.env").exists():
    load_dotenv("../../.env")

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY no encontrada")


def project_root() -> Path:
    nb_dir = Path(".").resolve()
    if nb_dir.name == "experiments":
        return nb_dir.parents[1]
    if nb_dir.name == "notebooks":
        return nb_dir.parent
    return nb_dir


def resolve_paths(runtime: str):
    if runtime == "colab":
        docs_dir = "/content/drive/MyDrive/RAG_UPC_Final_project"
        graph_dir = f"{docs_dir}/graph"
        chroma_path = f"{docs_dir}/chroma_db"
    else:
        root = project_root()
        docs_dir = str(root / "data" / "raw")
        graph_dir = str(root / "data" / "graph")
        chroma_path = str(root / "data" / "chroma_db")
    batch_dir = str(Path(graph_dir) / "batches")
    Path(graph_dir).mkdir(parents=True, exist_ok=True)
    Path(batch_dir).mkdir(parents=True, exist_ok=True)
    return docs_dir, graph_dir, batch_dir, chroma_path


DOCS_DIR, GRAPH_DIR, BATCH_DIR, CHROMA_PATH = resolve_paths(RUNTIME)
ROOT = project_root()

if RUNTIME == "local" and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"RUNTIME={RUNTIME}")
print(f"DOCS_DIR={DOCS_DIR}")
print(f"GRAPH_DIR={GRAPH_DIR}")
print(f"CHROMA_PATH={CHROMA_PATH}")
print(f"SCHEMA_VERSION={SCHEMA_VERSION}  CHUNK_STRATEGY={CHUNK_STRATEGY}")
print(f"BATCH_ID={BATCH_ID}  BATCH_SIZE={BATCH_SIZE}  RUN_ALL_BATCHES={RUN_ALL_BATCHES}")
print(f"FORCE_REGEN={FORCE_REGEN}  KG_CLUSTER={KG_CLUSTER}  FALLBACK={KG_CLUSTER_FALLBACK}")
print(f"REINDEX_CHROMA={REINDEX_CHROMA}  RUN_GRAPH_EXTRACT={RUN_GRAPH_EXTRACT}")
print(f"RUN_ALL_BATCHES={RUN_ALL_BATCHES}  SKIP_COMPLETED={SKIP_COMPLETED_BATCHES}  BATCH_IDS={BATCH_IDS}")


Mounted at /content/drive
RUNTIME=colab
DOCS_DIR=/content/drive/MyDrive/RAG_UPC_Final_project
GRAPH_DIR=/content/drive/MyDrive/RAG_UPC_Final_project/graph
CHROMA_PATH=/content/drive/MyDrive/RAG_UPC_Final_project/chroma_db
SCHEMA_VERSION=2  CHUNK_STRATEGY=recursive
BATCH_ID=6  BATCH_SIZE=10  RUN_ALL_BATCHES=False
FORCE_REGEN=False  KG_CLUSTER=True  FALLBACK=True
REINDEX_CHROMA=False  RUN_GRAPH_EXTRACT=True
RUN_ALL_BATCHES=False  SKIP_COMPLETED=True  BATCH_IDS=None


## 1. Utilidades ingest (dedup + chunk alineado)

In [26]:
TABLE_ROW_SEP = " || "


def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def deduplicate_filepaths(filepaths: list[str | Path]) -> list[Path]:
    """Un acta .doc + .docx → preferir .docx (igual que src/ingest.py)."""
    by_id: dict[str, Path] = {}

    def priority(fp: Path) -> int:
        low = fp.name.lower()
        if low.endswith(".docx"):
            return 0
        if low.endswith(".doc"):
            return 1
        return 2

    for fp in filepaths:
        p = Path(fp)
        doc_id = p.stem
        if doc_id not in by_id:
            by_id[doc_id] = p
            continue
        if priority(p) < priority(by_id[doc_id]):
            print(f"  dedup: {p.name} sustituye a {by_id[doc_id].name}")
            by_id[doc_id] = p
        else:
            print(f"  dedup: ignorado {p.name} (ya usamos {by_id[doc_id].name})")
    return sorted(by_id.values(), key=lambda x: x.name.lower())


def list_acta_files_dedup(raw_dir: str | Path) -> list[Path]:
    raw = Path(raw_dir)
    files = list(raw.glob("*.docx")) + list(raw.glob("*.doc"))
    if not files:
        raise FileNotFoundError(f"Sin .doc/.docx en {raw_dir}")
    return deduplicate_filepaths(files)


def is_bad_chunk(chunk: str, min_words: int = 40, min_alpha_ratio: float = 0.65) -> bool:
    words = len(chunk.split())
    if words < min_words:
        return True
    alpha_ratio = sum(c.isalpha() for c in chunk) / max(len(chunk), 1)
    return alpha_ratio < min_alpha_ratio


def chunk_text_recursive(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    overlap: int = CHUNK_OVERLAP,
    min_chunk_size: int = 50,
) -> list[str]:
    if not text:
        return []
    from langchain_text_splitters import RecursiveCharacterTextSplitter

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
        length_function=len,
    )
    raw_chunks = splitter.split_text(text)
    return [c for c in raw_chunks if len(c) >= min_chunk_size and not is_bad_chunk(c)]


def chunk_document(text: str, strategy: str = CHUNK_STRATEGY) -> list[str]:
    if strategy != "recursive":
        raise ValueError(f"Estrategia no soportada en v2: {strategy}")
    return chunk_text_recursive(text)


def chunk_id_for(doc_id: str, index: int) -> str:
    return f"{doc_id}__c{index:04d}"


def batch_json_path_v2(batch_id: int) -> Path:
    return Path(GRAPH_DIR) / f"graph_actas_e5_original_batch{batch_id:02d}_v2.json"


def batch_tag(batch_id: int) -> str:
    return f"batch{batch_id:02d}"


def batch_manifest_path(batch_id: int) -> Path:
    return Path(BATCH_DIR) / f"batch_{batch_id:02d}_manifest_v2.json"


print("✅ Utilidades ingest v2 cargadas (dedup + chunk recursive)")


✅ Utilidades ingest v2 cargadas (dedup + chunk recursive)


## 2. Extracción actas (docx / antiword)

In [27]:
from docx import Document
import subprocess as sp

EMBED_MODEL_NAME = "intfloat/multilingual-e5-base"


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    text = re.sub(r"[\u00ad\u2010-\u2015\u2212\uff0d]", "-", text)
    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)
    text = re.sub(r"[\u2022\u2023\u25cf\u25e6\u2043\u2219\u00b7]", "-", text)
    text = re.sub(r"(?m)^\s*\d{1,4}\s*$", "", text)
    text = re.sub(r"([.\-_=])\1{2,}", r"\1", text)
    text = re.sub(r"\s+([.,;:!?\)])", r"\1", text)
    text = re.sub(r"(?mi)^\s*(fecha|lugar|hora de inicio|hora de finalizacion|asistentes)\s*$", "", text)
    text = re.sub(r"(?mi)^\s*(firmado|firma).*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_docx(fp: Path) -> str | None:
    try:
        doc = Document(fp)
    except Exception as e:
        print(f"  ⚠️ {fp.name}: {e}")
        return None
    parts = []
    for p in doc.paragraphs:
        t = normalize_text(p.text)
        if t:
            parts.append(t)
    for table in doc.tables:
        for row in table.rows:
            row_texts, seen = [], set()
            for cell in row.cells:
                t = normalize_text(cell.text)
                if t and t not in seen:
                    seen.add(t)
                    row_texts.append(t)
            if row_texts:
                parts.append(TABLE_ROW_SEP.join(row_texts))
    return "\n".join(parts).strip() or None


def extract_doc_antiword(fp: Path) -> str | None:
    try:
        r = sp.run(["antiword", str(fp)], capture_output=True, text=True)
        return r.stdout.strip() if r.returncode == 0 else None
    except Exception as e:
        print(f"  ⚠️ antiword {fp.name}: {e}")
        return None


def extract_acta_text(fp: Path) -> str | None:
    raw = extract_docx(fp) if fp.suffix.lower() == ".docx" else extract_doc_antiword(fp)
    return clean_text(raw) if raw else None


def build_documents(acta_files: list[Path]) -> list[dict]:
    """Actas dedup → {doc_id, chunks, chunk_ids} alineado con Chroma."""
    documents = []
    for fp in acta_files:
        doc_id = fp.stem
        text = extract_acta_text(fp)
        if not text:
            print(f"  ⚠️ sin texto: {fp.name}")
            continue
        chunks = chunk_document(text, strategy=CHUNK_STRATEGY)
        if not chunks:
            print(f"  ⚠️ sin chunks: {fp.name}")
            continue
        chunk_ids = [chunk_id_for(doc_id, i) for i in range(len(chunks))]
        documents.append({"doc_id": doc_id, "chunks": chunks, "chunk_ids": chunk_ids})
    return documents


print("✅ Extracción actas (docx/antiword + clean_text) lista")


✅ Extracción actas (docx/antiword + clean_text) lista


## 3. Inventario corpus y batches

In [28]:
def batch_slice(files: list[Path], batch_id: int, batch_size: int) -> list[Path]:
    start = (batch_id - 1) * batch_size
    return files[start : start + batch_size]


all_actas = list_acta_files_dedup(RAW_DOCS_PATH_OVERRIDE or DOCS_DIR)
n_batches = (len(all_actas) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"Actas únicas (dedup): {len(all_actas)}")
print(f"BATCH_SIZE={BATCH_SIZE} → {n_batches} batches")
print(f"JSON pattern: graph_actas_e5_original_batch{{NN}}_v2.json")

for batch_id in range(1, n_batches + 1):
    batch_files = batch_slice(all_actas, batch_id, BATCH_SIZE)
    out = batch_json_path_v2(batch_id)
    exists = "✓" if out.exists() else " "
    print(f"  [{exists}] batch{batch_id:02d}: {len(batch_files)} actas → {out.name}")

if not RUN_ALL_BATCHES:
    batch_files = batch_slice(all_actas, BATCH_ID, BATCH_SIZE)
    if not batch_files:
        raise ValueError(f"Batch {BATCH_ID} vacío")
    manifest = {
        "schema_version": SCHEMA_VERSION,
        "batch_id": BATCH_ID,
        "batch_size": BATCH_SIZE,
        "tag": batch_tag(BATCH_ID),
        "sources": [f.stem for f in batch_files],
        "files": [str(f) for f in batch_files],
    }
    with open(batch_manifest_path(BATCH_ID), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"\nBatch activo: {BATCH_ID} ({len(batch_files)} actas)")
    for f in batch_files:
        print(f"  • {f.name}")


Actas únicas (dedup): 60
BATCH_SIZE=10 → 6 batches
JSON pattern: graph_actas_e5_original_batch{NN}_v2.json
  [✓] batch01: 10 actas → graph_actas_e5_original_batch01_v2.json
  [✓] batch02: 10 actas → graph_actas_e5_original_batch02_v2.json
  [✓] batch03: 10 actas → graph_actas_e5_original_batch03_v2.json
  [✓] batch04: 10 actas → graph_actas_e5_original_batch04_v2.json
  [✓] batch05: 10 actas → graph_actas_e5_original_batch05_v2.json
  [ ] batch06: 10 actas → graph_actas_e5_original_batch06_v2.json



Batch activo: 6 (10 actas)
  • 254275-DO-AVO-30-V01-260521_26.docx
  • 254275-DO-AVO-32-V01-260611.docx
  • Acta 1 Argañosa.docx
  • Acta 1 Aviles AC.docx
  • Acta 1. Arakaldo.docx
  • Acta 1. Balmaseda.docx
  • Acta 1. Itsasondo.docx
  • Acta 1. La Herrera.docx
  • Acta 1. Legorreta.docx
  • Nota_Técnica_Ejecucion_Radon_SIKA.docx


## 3b. Helpers batches (Fase 3)

In [29]:
def iter_batch_ids() -> list[int]:
    if BATCH_IDS is not None:
        return list(BATCH_IDS)
    if RUN_ALL_BATCHES:
        return list(range(1, n_batches + 1))
    return [BATCH_ID]


print(f"Fase 3 batches planificados: {iter_batch_ids()}")


Fase 3 batches planificados: [6]


## 4. Reindex Chroma (Fase 1 — `REINDEX_CHROMA=True`)

In [30]:
import json
import math
import time

import chromadb
import torch
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

CHUNKS_CATALOG_PATH = Path(GRAPH_DIR) / "chunks_catalog_v2.json"
BM25_OUT = Path(CHROMA_PATH) / "bm25.json"


class E5EmbedderNB:
    """E5 passage embeddings (compatible con prod)."""

    def __init__(self, model_name: str = EMBED_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.eval()

    def embed_passages(self, texts: list[str], batch_size: int = 16) -> list[list[float]]:
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [f"passage: {t}" for t in texts[i : i + batch_size]]
            inputs = self.tokenizer(batch, return_tensors="pt", truncation=True, padding=True)
            with torch.no_grad():
                hidden = self.model(**inputs).last_hidden_state[:, 0, :]
                hidden = torch.nn.functional.normalize(hidden, p=2, dim=1)
                out.extend(hidden.numpy().tolist())
        return out


class BM25Log1(BM25Okapi):
    def __init__(self, chunk_terms, chunk_ids, **kwargs):
        super().__init__(chunk_terms, **kwargs)
        self.chunk_ids = list(chunk_ids)
        df: dict[str, int] = {}
        for doc in self.doc_freqs:
            for term in set(doc):
                df[term] = df.get(term, 0) + 1
        n = self.corpus_size
        for term, dfi in df.items():
            self.idf[term] = math.log(1 + (n - dfi + 0.5) / (dfi + 0.5))

    @staticmethod
    def extract_terms(text: str) -> list[str]:
        return re.findall(r"\b[a-zA-ZáéíóúñÁÉÍÓÚÑ]+\b", text.lower())

    def save(self, path: Path) -> None:
        data = {
            "chunk_ids": self.chunk_ids,
            "idf": {k: float(v) for k, v in self.idf.items()},
            "doc_freqs": self.doc_freqs,
            "doc_len": self.doc_len,
            "avgdl": self.avgdl,
            "corpus_size": self.corpus_size,
            "k1": self.k1,
            "b": self.b,
        }
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps(data), encoding="utf-8")


def index_documents_to_chroma(documents: list[dict]) -> dict:
    all_texts, all_doc_ids, all_ids = [], [], []
    for doc in documents:
        for chunk, cid in zip(doc["chunks"], doc["chunk_ids"]):
            all_texts.append(chunk)
            all_doc_ids.append(doc["doc_id"])
            all_ids.append(cid)

    if len(all_ids) != len(set(all_ids)):
        raise ValueError("chunk_id duplicados — revisa dedup doc/docx")

    print(f"Indexando {len(all_texts)} chunks de {len(documents)} actas…")
    t0 = time.perf_counter()
    embedder = E5EmbedderNB()
    embeddings = embedder.embed_passages(all_texts)
    print(f"  embeddings: {(time.perf_counter()-t0)/60:.1f} min")

    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    collection = client.create_collection(
        name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
    )
    metadatas = [{"doc_id": d, "chunk_id": c} for d, c in zip(all_doc_ids, all_ids)]
    for i in range(0, len(all_texts), 5000):
        j = min(i + 5000, len(all_texts))
        collection.add(
            ids=all_ids[i:j],
            documents=all_texts[i:j],
            embeddings=embeddings[i:j],
            metadatas=metadatas[i:j],
        )

    terms = [BM25Log1.extract_terms(t) for t in all_texts]
    bm25 = BM25Log1(terms, all_ids)
    bm25.save(BM25_OUT)

    catalog = {
        "schema_version": SCHEMA_VERSION,
        "collection": COLLECTION_NAME,
        "chunk_strategy": CHUNK_STRATEGY,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "n_actas": len(documents),
        "n_chunks": len(all_ids),
        "documents": [
            {"doc_id": d["doc_id"], "chunk_ids": d["chunk_ids"], "n_chunks": len(d["chunk_ids"])}
            for d in documents
        ],
    }
    CHUNKS_CATALOG_PATH.write_text(
        json.dumps(catalog, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    print(f"✅ Chroma: {collection.count()} chunks → {CHROMA_PATH}/{COLLECTION_NAME}")
    print(f"✅ BM25 → {BM25_OUT}")
    print(f"✅ Catálogo chunks → {CHUNKS_CATALOG_PATH}")
    return catalog


if not REINDEX_CHROMA:
    print("REINDEX_CHROMA=False — salto reindex. Pon True para Fase 1.")
else:
    docs_for_index = build_documents(all_actas)
    if not docs_for_index:
        raise RuntimeError("Sin documentos para indexar")
    chunks_catalog = index_documents_to_chroma(docs_for_index)
    print(f"Actas indexadas: {chunks_catalog['n_actas']}")


REINDEX_CHROMA=False — salto reindex. Pon True para Fase 1.


## 5. Validación Chroma ↔ corpus

In [31]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)
try:
    collection = client.get_collection(COLLECTION_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Colección {COLLECTION_NAME!r} no encontrada en {CHROMA_PATH}. "
        "Ejecuta reindex (Fase 1) o REINDEX_CHROMA=True."
    ) from exc

data = collection.get(include=["metadatas"])
chunk_ids = data["ids"]
metas = data["metadatas"]
doc_ids_chroma = {m["doc_id"] for m in metas}
chunk_map = {cid: m for cid, m in zip(chunk_ids, metas)}

print(f"Chroma: {len(chunk_ids)} chunks, {len(doc_ids_chroma)} actas")
print(f"Colección: {COLLECTION_NAME}")

acta_stems = {p.stem for p in all_actas}
missing_in_chroma = acta_stems - doc_ids_chroma
extra_in_chroma = doc_ids_chroma - acta_stems

print(f"\nActas en corpus dedup: {len(acta_stems)}")
print(f"Actas sin chunks en Chroma: {len(missing_in_chroma)}")
if missing_in_chroma:
    for d in sorted(missing_in_chroma)[:10]:
        print(f"  ⚠️  {d}")
    if len(missing_in_chroma) > 10:
        print(f"  ... +{len(missing_in_chroma) - 10} más")

print(f"Actas en Chroma no en corpus raw: {len(extra_in_chroma)}")
if extra_in_chroma and len(extra_in_chroma) <= 5:
    for d in sorted(extra_in_chroma):
        print(f"  ℹ️  {d}")

# Muestra chunk_id format
sample = chunk_ids[:3]
print(f"\nEjemplo chunk_id: {sample}")


Chroma: 1544 chunks, 60 actas
Colección: cosora_actas_e5

Actas en corpus dedup: 60
Actas sin chunks en Chroma: 0
Actas en Chroma no en corpus raw: 0

Ejemplo chunk_id: ['244170-DF-INF_TEC_FT-01-V01_1004_Panelsandwich__c0000', '244170-DF-INF_TEC_FT-01-V01_1004_Panelsandwich__c0001', '244170-DF-INF_TEC_FT-01-V01_1004_Panelsandwich__c0002']


### 5b. Exportar `chunks_catalog_v2.json` (sin reindex)

Si ya tienes Chroma en Drive pero **no** ves `graph/chunks_catalog_v2.json`, ejecuta esta celda.
Solo lee metadatos de Chroma y escribe el JSON en `GRAPH_DIR`.

In [ ]:
EXPORT_CHUNKS_CATALOG = True  # False para saltar

CHUNKS_CATALOG_PATH = Path(GRAPH_DIR) / "chunks_catalog_v2.json"

if not EXPORT_CHUNKS_CATALOG:
    print("EXPORT_CHUNKS_CATALOG=False — salto")
else:
    by_doc: dict[str, list[str]] = {}
    for cid, meta in zip(chunk_ids, metas):
        doc_id = meta.get("doc_id") or "unknown"
        by_doc.setdefault(doc_id, []).append(cid)
    for doc_id in by_doc:
        by_doc[doc_id].sort()

    catalog = {
        "schema_version": SCHEMA_VERSION,
        "collection": COLLECTION_NAME,
        "chunk_strategy": CHUNK_STRATEGY,
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "n_actas": len(by_doc),
        "n_chunks": len(chunk_ids),
        "exported_from_chroma": True,
        "documents": [
            {"doc_id": d, "chunk_ids": by_doc[d], "n_chunks": len(by_doc[d])}
            for d in sorted(by_doc)
        ],
    }
    CHUNKS_CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHUNKS_CATALOG_PATH.write_text(
        json.dumps(catalog, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(f"✅ Catálogo exportado → {CHUNKS_CATALOG_PATH}")
    print(f"   {catalog['n_actas']} actas, {catalog['n_chunks']} chunks")

## 6. Schema JSON v2

In [32]:
def relation_v2(
    subject: str,
    predicate: str,
    obj: str,
    *,
    source_doc: str,
    source_chunk_id: str,
    source_docs: list | None = None,
    source_chunk_ids: list | None = None,
) -> dict:
    return {
        "subject": subject,
        "predicate": predicate,
        "object": obj,
        "source_doc": source_doc,
        "source_chunk_id": source_chunk_id,
        "source_docs": source_docs or [],
        "source_chunk_ids": source_chunk_ids or [],
    }


def graph_to_dict_v2(
    relations: list[dict],
    *,
    sources: list[str],
    batch_id: int,
    n_chunks: int,
    entity_clusters: dict | None = None,
    edge_clusters: dict | None = None,
) -> dict:
    entities = sorted({r["subject"] for r in relations} | {r["object"] for r in relations})
    predicates = sorted({r["predicate"] for r in relations})
    return {
        "entities": entities,
        "edges": predicates,
        "relations": relations,
        "entity_clusters": entity_clusters or {},
        "edge_clusters": edge_clusters or {},
        "meta": {
            "schema_version": SCHEMA_VERSION,
            "model": KG_MODEL,
            "kg_source": KG_SOURCE,
            "batch_id": batch_id,
            "batch_size": BATCH_SIZE,
            "tag": batch_tag(batch_id),
            "n_inputs": n_chunks,
            "sources": sources,
            "context": KG_CONTEXT,
            "cluster": KG_CLUSTER,
            "chunk_strategy": CHUNK_STRATEGY,
        },
    }


print("✅ Schema v2 helpers listos (graph_to_dict_v2, relation_v2)")


✅ Schema v2 helpers listos (graph_to_dict_v2, relation_v2)


## 7. KGGen por chunk (Fase 2 — piloto) / Fase 3 — todos los batches

In [33]:
import time
from collections import defaultdict

from kg_gen import KGGen


def load_batch_documents_from_chroma(batch_files: list[Path]) -> list[dict]:
    """Carga chunks del batch desde Chroma (texto + chunk_id)."""
    import chromadb

    target_docs = {fp.stem for fp in batch_files}
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    collection = client.get_collection(COLLECTION_NAME)
    data = collection.get(include=["documents", "metadatas"])
    by_doc: dict[str, list[tuple[str, str]]] = defaultdict(list)
    for text, meta in zip(data["documents"], data["metadatas"]):
        doc_id = meta["doc_id"]
        if doc_id in target_docs:
            by_doc[doc_id].append((meta["chunk_id"], text))
    documents = []
    for fp in batch_files:
        doc_id = fp.stem
        pairs = sorted(by_doc.get(doc_id, []), key=lambda x: x[0])
        if not pairs:
            print(f"  ⚠️  sin chunks Chroma: {doc_id}")
            continue
        documents.append({
            "doc_id": doc_id,
            "chunk_ids": [p[0] for p in pairs],
            "chunks": [p[1] for p in pairs],
        })
    return documents


def load_batch_documents(batch_files: list[Path]) -> list[dict]:
    """Chroma si existe; si no, chunking local (mismo algoritmo que Fase 1)."""
    try:
        docs = load_batch_documents_from_chroma(batch_files)
        if docs:
            return docs
    except Exception as exc:
        print(f"  Chroma no disponible ({exc}) — chunking local")
    return build_documents(batch_files)


def build_entity_map(entity_clusters: dict | None) -> dict[str, str]:
    mapping: dict[str, str] = {}
    for canonical, members in (entity_clusters or {}).items():
        for name in members:
            mapping[name.lower()] = canonical
        mapping[str(canonical).lower()] = canonical
    return mapping


def map_entity(name: str, entity_map: dict[str, str]) -> str:
    return entity_map.get(name.lower(), name)


def attach_provenance_to_graph(graph, pre_cluster_rels: list[dict]) -> list[dict]:
    entity_map = build_entity_map(
        {k: sorted(list(v)) for k, v in (graph.entity_clusters or {}).items()}
        if graph.entity_clusters
        else {}
    )
    prov_index: dict[tuple, dict] = defaultdict(lambda: {"source_docs": set(), "source_chunk_ids": set()})
    for r in pre_cluster_rels:
        ms = map_entity(r["subject"], entity_map)
        mo = map_entity(r["object"], entity_map)
        key = (ms.lower(), r["predicate"].lower(), mo.lower())
        prov_index[key]["source_docs"].add(r["source_doc"])
        prov_index[key]["source_chunk_ids"].add(r["source_chunk_id"])

    final: list[dict] = []
    for s, p, o in graph.relations:
        key = (s.lower(), p.lower(), o.lower())
        prov = prov_index.get(key)
        if not prov:
            # fallback: buscar por sujeto+objeto sin predicado exacto
            for (ps, pp, po), pv in prov_index.items():
                if ps == s.lower() and po == o.lower():
                    prov = pv
                    break
        docs = sorted(prov["source_docs"]) if prov else []
        chunks = sorted(prov["source_chunk_ids"]) if prov else []
        final.append(
            relation_v2(
                s, p, o,
                source_doc=docs[0] if docs else "",
                source_chunk_id=chunks[0] if chunks else "",
                source_docs=docs,
                source_chunk_ids=chunks,
            )
        )
    return final


def extract_batch_graph_v2(batch_id: int, batch_files: list[Path]) -> tuple[dict, dict]:
    out_path = batch_json_path_v2(batch_id)
    if out_path.exists() and not FORCE_REGEN:
        print(f"📁 {out_path.name} existe — skip (FORCE_REGEN=False)")
        with open(out_path, encoding="utf-8") as f:
            gd = json.load(f)
        return gd, {
            "batch_id": batch_id,
            "skipped": True,
            "n_triples": len(gd.get("relations", [])),
            "json_file": out_path.name,
        }

    documents = load_batch_documents(batch_files)
    if not documents:
        raise RuntimeError(f"Batch {batch_id}: sin documentos/chunks")

    n_chunks = sum(len(d["chunks"]) for d in documents)
    sources = [d["doc_id"] for d in documents]
    print(f"Batch {batch_id}: {len(documents)} actas, {n_chunks} chunks → KGGen…")

    kg = KGGen(model=KG_MODEL, temperature=0.0, api_key=api_key)
    partial_graphs = []
    pre_cluster_rels: list[dict] = []
    t0 = time.perf_counter()
    done = 0

    for doc in documents:
        for chunk_text, chunk_id in zip(doc["chunks"], doc["chunk_ids"]):
            done += 1
            ti = time.perf_counter()
            g_i = kg.generate(
                input_data=chunk_text,
                context=KG_CONTEXT,
                chunk_size=KG_CHUNK_SIZE,
                cluster=False,
            )
            partial_graphs.append(g_i)
            for s, p, o in g_i.relations:
                pre_cluster_rels.append(
                    relation_v2(
                        s, p, o,
                        source_doc=doc["doc_id"],
                        source_chunk_id=chunk_id,
                    )
                )
            print(
                f"  {done:>4}/{n_chunks} {chunk_id[:44]:<44} "
                f"+{len(g_i.relations):>3} ({time.perf_counter()-ti:.1f}s)"
            )

    print(f"Agregando {len(partial_graphs)} grafos parciales…")
    graph = kg.aggregate(partial_graphs)
    if KG_CLUSTER and graph.relations:
        print("Clustering global (KG_CLUSTER=True)…")
        try:
            graph = kg.cluster(graph, context=KG_CONTEXT)
        except Exception as exc:
            if KG_CLUSTER_FALLBACK:
                print(f"⚠️  cluster falló ({type(exc).__name__}) — sigue sin cluster")
                print(f"   {str(exc)[:200]}")
            else:
                raise

    relations = attach_provenance_to_graph(graph, pre_cluster_rels)
    entity_clusters = {
        k: sorted(list(v)) for k, v in (graph.entity_clusters or {}).items()
    }
    edge_clusters = {
        k: sorted(list(v)) for k, v in (graph.edge_clusters or {}).items()
    }
    gd = graph_to_dict_v2(
        relations,
        sources=sources,
        batch_id=batch_id,
        n_chunks=n_chunks,
        entity_clusters=entity_clusters,
        edge_clusters=edge_clusters,
    )

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(gd, f, ensure_ascii=False, indent=2)

    manifest = {
        "schema_version": SCHEMA_VERSION,
        "batch_id": batch_id,
        "tag": batch_tag(batch_id),
        "json_file": out_path.name,
        "sources": sources,
        "n_triples": len(relations),
        "n_chunks": n_chunks,
    }
    with open(batch_manifest_path(batch_id), "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    elapsed = (time.perf_counter() - t0) / 60
    print(f"✅ batch{batch_id:02d}: {len(relations)} triples en {elapsed:.1f} min → {out_path}")
    return gd, {"batch_id": batch_id, "skipped": False, "n_triples": len(relations), "n_chunks": n_chunks, "minutes": round(elapsed, 1), "json_file": out_path.name}


regen_results: list[dict] = []

if not RUN_GRAPH_EXTRACT:
    print("RUN_GRAPH_EXTRACT=False — salto Fase 2/3.")
else:
    planned = iter_batch_ids()
    print(f"Fase 3: procesando {len(planned)} batch(es): {planned}")
    for bid in planned:
        files = batch_slice(all_actas, bid, BATCH_SIZE)
        if not files:
            print(f"Batch {bid} vacío — skip")
            regen_results.append({"batch_id": bid, "skipped": True, "reason": "empty_batch"})
            continue
        out_path = batch_json_path_v2(bid)
        if SKIP_COMPLETED_BATCHES and out_path.exists() and not FORCE_REGEN:
            with open(out_path, encoding="utf-8") as f:
                gd_skip = json.load(f)
            n = len(gd_skip.get("relations", []))
            print(f"⏭️  batch{bid:02d}: {out_path.name} existe ({n} triples) — skip")
            regen_results.append({"batch_id": bid, "skipped": True, "n_triples": n, "json_file": out_path.name})
            continue
        _, stats = extract_batch_graph_v2(bid, files)
        regen_results.append(stats)

    done = sum(1 for r in regen_results if not r.get("skipped"))
    skipped = sum(1 for r in regen_results if r.get("skipped"))
    total_triples = sum(r.get("n_triples", 0) for r in regen_results)
    print(f"\n📊 Regen: {done} generados, {skipped} skip, {total_triples} triples total")


Fase 3: procesando 1 batch(es): [6]
Batch 6: 10 actas, 228 chunks → KGGen…
     1/228 254275-DO-AVO-30-V01-260521_26__c0000        +  7 (5.9s)
     2/228 254275-DO-AVO-30-V01-260521_26__c0001        +  7 (5.7s)
     3/228 254275-DO-AVO-30-V01-260521_26__c0002        +  6 (6.5s)
     4/228 254275-DO-AVO-30-V01-260521_26__c0003        + 11 (8.1s)
     5/228 254275-DO-AVO-30-V01-260521_26__c0004        +  8 (6.8s)
     6/228 254275-DO-AVO-30-V01-260521_26__c0005        +  6 (5.3s)
     7/228 254275-DO-AVO-30-V01-260521_26__c0006        +  6 (0.0s)
     8/228 254275-DO-AVO-30-V01-260521_26__c0007        +  3 (8.3s)
     9/228 254275-DO-AVO-30-V01-260521_26__c0008        +  3 (0.0s)
    10/228 254275-DO-AVO-30-V01-260521_26__c0009        +  8 (0.0s)
    11/228 254275-DO-AVO-30-V01-260521_26__c0010        +  8 (0.0s)
    12/228 254275-DO-AVO-30-V01-260521_26__c0011        +  7 (0.0s)
    13/228 254275-DO-AVO-30-V01-260521_26__c0012        +  6 (0.0s)
    14/228 254275-DO-AVO-30-V01-260521_26

## 8. Validación grafo v2

In [34]:
def validate_graph_v2(gd: dict, chunk_map: dict | None = None) -> dict:
    rels = gd.get("relations", [])
    n = len(rels)
    with_doc = sum(1 for r in rels if r.get("source_doc"))
    with_chunk = sum(1 for r in rels if r.get("source_chunk_id"))
    missing_chunk = [
        r["source_chunk_id"]
        for r in rels
        if r.get("source_chunk_id") and chunk_map is not None and r["source_chunk_id"] not in chunk_map
    ]
    return {
        "n_triples": n,
        "pct_source_doc": round(100 * with_doc / n, 1) if n else 0,
        "pct_source_chunk_id": round(100 * with_chunk / n, 1) if n else 0,
        "missing_in_chroma": len(missing_chunk),
    }


if RUN_GRAPH_EXTRACT:
    try:
        import chromadb
        _client = chromadb.PersistentClient(path=CHROMA_PATH)
        _col = _client.get_collection(COLLECTION_NAME)
        _chunk_map = {cid: m for cid, m in zip(_col.get()["ids"], _col.get()["metadatas"])}
    except Exception:
        _chunk_map = None

    check_ids = iter_batch_ids() if RUN_GRAPH_EXTRACT else ([BATCH_ID] if not RUN_ALL_BATCHES else range(1, n_batches + 1))
    for bid in check_ids:
        p = batch_json_path_v2(bid)
        if not p.exists():
            print(f"  batch{bid:02d}: JSON no generado")
            continue
        with open(p, encoding="utf-8") as f:
            _gd = json.load(f)
        stats = validate_graph_v2(_gd, _chunk_map)
        ok = stats["pct_source_chunk_id"] == 100 and stats["missing_in_chroma"] == 0
        mark = "✅" if ok else "⚠️"
        print(
            f"{mark} batch{bid:02d}: {stats['n_triples']} triples | "
            f"doc={stats['pct_source_doc']}% chunk={stats['pct_source_chunk_id']}% | "
            f"missing_chroma={stats['missing_in_chroma']}"
        )


⚠️ batch06: 1512 triples | doc=99.9% chunk=99.9% | missing_chroma=0


## 8b. Reporte Fase 3 → `regen_report_v2.json`

In [35]:
REGEN_REPORT_PATH = Path(GRAPH_DIR) / "regen_report_v2.json"

if RUN_GRAPH_EXTRACT and regen_results:
    report = {
        "schema_version": SCHEMA_VERSION,
        "n_batches_corpus": n_batches,
        "n_actas_dedup": len(all_actas),
        "run_all_batches": RUN_ALL_BATCHES,
        "batch_ids_filter": BATCH_IDS,
        "force_regen": FORCE_REGEN,
        "skip_completed_batches": SKIP_COMPLETED_BATCHES,
        "batches": regen_results,
        "n_generated": sum(1 for r in regen_results if not r.get("skipped")),
        "n_skipped": sum(1 for r in regen_results if r.get("skipped")),
        "n_triples_total": sum(r.get("n_triples", 0) for r in regen_results),
    }
    REGEN_REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"📄 regen_report_v2.json → {REGEN_REPORT_PATH}")
    print(f"   generados={report['n_generated']}  skip={report['n_skipped']}  triples={report['n_triples_total']}")
else:
    print("Ejecuta §7 (RUN_GRAPH_EXTRACT=True) para generar regen_report_v2.json")


📄 regen_report_v2.json → /content/drive/MyDrive/RAG_UPC_Final_project/graph/regen_report_v2.json
   generados=1  skip=0  triples=1512


## 8c. Catalog v2 → `catalog.json`

In [36]:
def refresh_catalog_v2(graph_dir: str | Path) -> dict:
    graph_dir = Path(graph_dir)
    batches = []
    for p in sorted(graph_dir.glob("graph_actas_e5_original_batch*_v2.json")):
        m = re.search(r"batch(\d+)_", p.name)
        if not m:
            continue
        bid = int(m.group(1))
        with open(p, encoding="utf-8") as f:
            gd = json.load(f)
        meta = gd.get("meta", {})
        batches.append({
            "batch_id": bid,
            "tag": meta.get("tag") or batch_tag(bid),
            "json_file": p.name,
            "schema_version": meta.get("schema_version", SCHEMA_VERSION),
            "sources": meta.get("sources") or [],
            "n_triples": len(gd.get("relations", [])),
            "n_chunks": meta.get("n_inputs", 0),
        })
    catalog = {"schema_version": SCHEMA_VERSION, "batches": batches}
    out = graph_dir / "catalog.json"
    out.write_text(json.dumps(catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"📋 catalog.json v2: {len(batches)} batches → {out}")
    return catalog


if RUN_GRAPH_EXTRACT:
    catalog_v2 = refresh_catalog_v2(GRAPH_DIR)
    for b in catalog_v2["batches"]:
        print(f"  • {b['tag']}: {b['n_triples']} triples ← {b['json_file']}")
else:
    print("Genera grafos (§7) para refrescar catalog.json")


📋 catalog.json v2: 6 batches → /content/drive/MyDrive/RAG_UPC_Final_project/graph/catalog.json
  • batch01: 1288 triples ← graph_actas_e5_original_batch01_v2.json
  • batch02: 1195 triples ← graph_actas_e5_original_batch02_v2.json
  • batch03: 1342 triples ← graph_actas_e5_original_batch03_v2.json
  • batch04: 1467 triples ← graph_actas_e5_original_batch04_v2.json
  • batch05: 1944 triples ← graph_actas_e5_original_batch05_v2.json
  • batch06: 1512 triples ← graph_actas_e5_original_batch06_v2.json


## 10. Subir Chroma v2 → GCS (deploy Cloud Run)

Ejecuta **una vez** tras §4/§5 para alinear GCS con el mismo índice que Aura (1544 chunks).

Requisito: autenticación GCP en Colab (`auth.authenticate_user()`).

In [ ]:
UPLOAD_CHROMA_TO_GCS = True  # False para saltar

if UPLOAD_CHROMA_TO_GCS:
    from google.colab import auth
    auth.authenticate_user()

    import os
    from pathlib import Path
    from google.cloud import storage
    import chromadb

    def upload_folder_to_gcs(bucket_name: str, source_folder: Path, prefix: str) -> int:
        client = storage.Client()
        bucket = client.bucket(bucket_name)
        count = 0
        for local_path in source_folder.rglob("*"):
            if not local_path.is_file():
                continue
            rel = local_path.relative_to(source_folder).as_posix()
            blob_path = f"{prefix.rstrip('/')}/{rel}"
            bucket.blob(blob_path).upload_from_filename(str(local_path))
            count += 1
            if count % 10 == 0:
                print(f"  ↑ {count} archivos...")
        return count

    bucket = os.getenv("GCP_BUCKET_NAME", "rag-actas-db-bucket")
    local_chroma = Path(CHROMA_PATH)
    counts = chromadb.PersistentClient(path=str(local_chroma)).get_collection(COLLECTION_NAME).count()
    print(f"Drive Chroma: {counts} chunks en {COLLECTION_NAME!r}")

    n = upload_folder_to_gcs(bucket, local_chroma, "chroma_db")
    print(f"✅ Subidos {n} archivos → gs://{bucket}/chroma_db/")
    print("Siguiente: en PC → python src/upload_chroma.py --verify-gcs && deploy.ps1")
else:
    print("UPLOAD_CHROMA_TO_GCS=False — salto")

Drive Chroma: 1544 chunks en 'cosora_actas_e5'


## 9. Siguiente paso

1. Revisar validación §8 (100% provenance, 0 missing en Chroma)
2. Abrir **`neo4j_graph_rag_v2.ipynb`** → carga Neo4j + rutas Cypher
3. Probar queries en Colab antes de promover a `src/`
